In [ ]:
# import packages
import pandas as pd
import numpy as np
import json, requests, os, dotenv
import geopy, folium, matplotlib

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.cluster import KMeans

# Get Data

In [ ]:
# get the original data from the Wikipedia
Toronto_df = pd.read_html("https://en.wikipedia.org/w/index.php?title=List_of_postal_codes_of_Canada:_M&oldid=926306543")[0]

# drop the unassigned rows
Toronto_df = Toronto_df[Toronto_df['Borough'] != 'Not assigned']

# combine neighbourhood
Toronto_df['Neighbourhood'] = Toronto_df.groupby(['Postcode'])['Neighbourhood'].transform(lambda x: ', '.join(x))

# drop the duplicate rows and reset the index
Toronto_df = Toronto_df.drop_duplicates().sort_values(by=['Postcode']).reset_index(drop=True)

In [ ]:
# read the data from website: latitude and longitude
df_cor = pd.read_csv('https://cocl.us/Geospatial_data')

# get the new dataframe with the latitude and longitude
Toronto_df_geo = Toronto_df.merge(df_cor, left_on = 'Postcode', right_on = 'Postal Code').drop(['Postal Code'], axis=1)
Toronto_df_geo.columns = [i.lower() for i in Toronto_df_geo.columns]
Toronto_df_geo.head(3)

# Clustering Data Preparation

In [ ]:
# # get the latitude and longtitude of Toronto, Canada
# address = 'Toronto, Canada'

# geolocator = geopy.geocoders.Nominatim(user_agent = "ny_explorer")
# latitude, longitude = geolocator.geocode(address).latitude, geolocator.geocode(address).longitude
# print(f'The geograpical coordinate of Toronto are {latitude}, {longitude}.')

In [ ]:
# create map of Toronto using latitude and longitude values
map_toronto = folium.Map(location=[Toronto_df_geo['latitude'].mean(), Toronto_df_geo['longitude'].mean()], zoom_start = 11)

# add markers to map
for lat, lng, postcode, neighbourhood in zip(Toronto_df_geo['latitude'], Toronto_df_geo['longitude'], Toronto_df_geo['postcode'], Toronto_df_geo['neighbourhood']):
    folium.CircleMarker([lat, lng], 
                        radius = 5,
                        tooltip = f"{postcode}: {neighbourhood}",
                        color = 'blue', fill = True, fill_color = '#3186cc', fill_opacity = 0.7).add_to(map_toronto)  
map_toronto

In [ ]:
# import the api_key for Foursquare
dotenv.load_dotenv("../personal_envs/neighborhood-preference-prediction.env", override=True)
api_key = os.getenv("toronto_venue_clustering")

In [ ]:
# write the function of getting venues from Foursquare
def get_foursquare_raw_data(api_key, lat, long, radius, limit):
    url = "https://api.foursquare.com/v3/places/search"
    headers = {"Accept": "application/json", "Authorization": f"Bearer {api_key}"}
    params = {"ll": f"{lat},{long}", "radius": radius, "limit": limit, "sort": "DISTANCE"}
    response = requests.get(url, headers = headers, params = params)
    response.raise_for_status()
    return response.json()['results']

In [ ]:
# get the venue list
venues_list = []

for postcode, lat, lng in zip(Toronto_df_geo['postcode'], Toronto_df_geo['latitude'], Toronto_df_geo['longitude']):
    
    raw_data = get_foursquare_raw_data(api_key, lat, lng, 1000, 50)
    for v in raw_data:
        venues_list.append({
            "postcode": postcode,
            "fsq_id": v.get('fsq_id', None),
            "name": v.get('name', None),
            "lat": v['geocodes']['main']['latitude'] if 'geocodes' in v else None,
            "lon": v['geocodes']['main']['longitude'] if 'geocodes' in v else None,
            "distance": v.get('distance', None),
            "category": [c['name'] for c in v.get('categories', [])] if v.get('categories') else [],
            "location_address": v['location'].get('address', None),
            "location_locality": v['location'].get('locality', None),
            "location_region": v['location'].get('region', None),
            "location_postcode": v['location'].get('postcode', None),
            "location_country": v['location'].get('country', None),
            "location_formatted_address": v['location'].get('formatted_address', None),
            "timezone": v.get('timezone', None),
            "likely_open": v.get('closed_bucket', None)
        })

df = pd.DataFrame(venues_list)

In [ ]:
# get the dummy data of category
mlb = MultiLabelBinarizer()
category_dummies = pd.DataFrame(mlb.fit_transform(df['category']), columns=mlb.classes_, index=df.index)
print(f'There are {len(category_dummies.columns)} uniques categories.')

Toronto_venues_dummy = pd.concat([df[['postcode']], category_dummies], axis=1)

In [ ]:
# get the dummy data grouped
Toronto_venues_dummy_group = Toronto_venues_dummy.groupby('postcode').mean().reset_index()

In [ ]:
# combine geo data with venues dummy data
Toronto_combined = Toronto_df_geo.merge(Toronto_venues_dummy_group, on = 'postcode')
Toronto_combined.head(3)

# Clustering

In [ ]:
# set number of clusters
kclusters = 10

# set the data for running the clustering
Toronto_clustering_data = Toronto_combined[Toronto_combined.columns[6: ]]

# run k-means clustering
kmeans = KMeans(n_clusters = kclusters, random_state = 0, n_init = 10).fit(Toronto_clustering_data)

# get the cuslter label
Toronto_combined['cluster_label'] = kmeans.labels_

In [ ]:
# create map
map_cluster = folium.Map(location=[Toronto_combined['latitude'].mean(), Toronto_combined['longitude'].mean()], zoom_start = 11)

# set color scheme for the clusters
x = np.arange(kclusters)
ys = [i + x + (i*x)**2 for i in range(kclusters)]
colors_array = matplotlib.cm.rainbow(np.linspace(0, 1, len(ys)))
rainbow = [matplotlib.colors.rgb2hex(i) for i in colors_array]

# add markers to the map
markers_colors = []
for lat, lon, postcode, cluster in zip(Toronto_combined['latitude'], Toronto_combined['longitude'], Toronto_combined['postcode'], Toronto_combined['cluster_label']):
    folium.CircleMarker([lat, lon], 
                        radius = 5, 
                        tooltip = f"{postcode}: Cluster {cluster}",
                        color = rainbow[cluster-1], fill = True, fill_color = rainbow[cluster-1], fill_opacity = 0.7).add_to(map_cluster)
       
map_cluster